In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
import json
import re 
import math
import concurrent.futures
from pymongo import MongoClient
import os
import json
from datetime import datetime
def clean_price_to_float(raw_string):
    """Cleans Swiss price formats (1.-, 16.–, 1.−) into sortable floats."""
    if not raw_string:
        return None
    cleaned = raw_string.strip()
    # Regex: Look for a dot followed by any non-digit chars at the end and replace with .00
    cleaned = re.sub(r'\.\D+$', '.00', cleaned)
    try:
        return float(cleaned)
    except ValueError:
        return cleaned

def parse_product(article):
    """Parses a single product card into a dictionary."""
    product_dict = {}
    
    # ID
    link_tag = article.find('a', {'data-testid': 'product-link'})
    product_dict['id'] = link_tag.get('href').rstrip('/').split('/')[-1] if link_tag else None
    
    # Brand & Name
    brand = article.find('span', class_='name')
    product_dict['brand'] = brand.text.strip() if brand else None
    name = article.find('span', attrs={'data-testid': lambda x: x and x.startswith('product-name')})
    product_dict['name'] = name.text.strip() if name else None
    
    # Price
    price_tag = article.find('span', {'data-testid': 'current-price'})
    product_dict['price'] = clean_price_to_float(price_tag.text) if price_tag else None
        
    # Quantity (Multipack Math)
    quantity_tag = article.find('span', {'data-testid': 'default-product-size'})
    if quantity_tag:
        raw_qty = quantity_tag.text.strip()
        multipack_match = re.search(r'(\d+)\s*[xX]\s*(\d+(?:\.\d+)?)\s*([a-zA-Z]+)', raw_qty)
        if multipack_match:
            total = int(multipack_match.group(1)) * float(multipack_match.group(2))
            unit = multipack_match.group(3)
            product_dict['quantity'] = f"{int(total) if total.is_integer() else total}{unit}"
        else:
            product_dict['quantity'] = raw_qty
    else:
        product_dict['quantity'] = None
    
    # Price per Unit & Unit
    product_dict['price_per_unit'], product_dict['unit'] = None, None
    ppu_tag = article.find('span', id=lambda x: x and x.endswith('-price-unit'))
    if ppu_tag:
        raw_ppu = ppu_tag.text.strip() 
        if '/' in raw_ppu:
            ppu_part, unit_part = raw_ppu.split('/', 1)
            product_dict['price_per_unit'] = clean_price_to_float(ppu_part)
            product_dict['unit'] = unit_part.strip() 
        else:
            product_dict['price_per_unit'] = clean_price_to_float(raw_ppu)
    
    # Image URL
    img = article.find('img')
    product_dict['image_url'] = img.get('src') if img else None
    
    return product_dict


In [55]:
def create_driver():
    print("Setting up the browser...")
    chrome_options = Options()
    chrome_options.add_argument("--headless=new") 
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver


def get_remaining_products_count(driver,url):
    rendered_html = get_rendered_page_sync(driver, url)
    soup = BeautifulSoup(rendered_html, 'html.parser')
    remaining_div = soup.find('div', class_='remaining-products')
    if remaining_div:
        raw_text = remaining_div.text.strip() 
        clean_numbers = re.sub(r'\D', '', raw_text) 
        if clean_numbers:
            return math.ceil(int(clean_numbers) / 100)

    return 0

# --- 2. The Selenium Scraper ---
def get_rendered_page_sync(driver,url): 
    try:
        print(f"Loading {url}...")
        driver.get(url)
        
        # Give Angular enough time to build the list of products
        print("Waiting for products to load...")
        time.sleep(2) 
        
        html_content = driver.page_source
        return html_content
    except Exception as e:
        print(f"Error loading page: {e}")
        return None


def scrape_single_page(url):
    """This function runs in parallel. It MUST create its own driver."""
    print(f"Thread starting for: {url}")
    
    # Create a unique driver for this specific thread
    driver = create_driver()
    page_products = []
    
    try:
        # Step A: Get the HTML
        rendered_html = get_rendered_page_sync(driver, url)
        
        if rendered_html:
            soup = BeautifulSoup(rendered_html, 'html.parser')
            
            # Step C: Find ALL product cards
            product_cards = soup.find_all('article', class_='product-card')
            
            # Step D: Loop through them and parse
            for card in product_cards:
                parsed_data = parse_product(card)
                if parsed_data and parsed_data.get('name'):
                    page_products.append(parsed_data)
                    
        return page_products # Return the list of dictionaries for this page

    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return []
        
    finally:
        # CRITICAL: Always close the thread's driver so you don't run out of RAM!
        driver.quit()



def fetch_product_data(base_url, path_to_json="migros_products.json", max_workers=1, fetch_online=True):
    """
    Fetches product data either by scraping Migros or loading a local JSON file.
    """
    
    # --- LOCAL FILE OVERRIDE ---
    if not fetch_online:
        print(f"Bypassing online scrape. Attempting to load local file: '{path_to_json}'...")
        
        if os.path.exists(path_to_json):
            with open(path_to_json, 'r', encoding='utf-8') as f:
                all_products = json.load(f)
            print(f"✅ Successfully loaded {len(all_products)} products from local JSON.")
            return all_products 
        else:
            print(f"⚠️ Warning: Local file '{path_to_json}' not found! Forcing online fetch...")
           

    # --- LIVE SCRAPING LOGIC ---
    print(f"Fetching online data from: {base_url}")

    temp_driver = create_driver()  
    # Calculate pages
    number_of_pages = get_remaining_products_count(temp_driver, base_url) + 1
    temp_driver.quit() 

    # Generate all URLs
    urls = [f"{base_url}?page={i}" for i in range(1, int(number_of_pages) + 1)]
    print(f"Total pages to scrape: {len(urls)}")
    all_products = []
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = executor.map(scrape_single_page, urls)
        for page_results in results:
            if page_results:
                all_products.extend(page_results)

    print(f"\nSuccessfully extracted {len(all_products)} total products.")

    # Save the fresh data to JSON
    with open(path_to_json, "w", encoding="utf-8") as f:
        json.dump(all_products, f, indent=4, ensure_ascii=False)
    print(f"All data successfully saved to '{path_to_json}'!")
    

    return all_products









In [56]:


def save_to_mongodb(products_list, db_name="migros_db", collection_name="products"):
    """Saves a list of dictionaries to MongoDB as daily snapshots."""
    
    # Connect to the local MongoDB server
    client = MongoClient("mongodb://localhost:27017/")
    db = client[db_name]
    collection = db[collection_name]
    
    print(f"Connected to MongoDB. Saving {len(products_list)} product snapshots...")
    
    # Capture the exact time this snapshot is being taken
    scrape_timestamp = datetime.utcnow()
    
    for product in products_list:
        if not product.get('id'):
            continue
            
        # 1. Add the timestamp so you can group/sort by date later
        product['scraped_at'] = scrape_timestamp
        
        # 2. We DO NOT set product['_id']. 
        # By leaving '_id' out, MongoDB will auto-generate a unique ObjectId for this snapshot.
        
        # 3. Use insert_one instead of update_one/upsert to append a new history record
        collection.insert_one(product)
        
    print("Snapshot data successfully saved to MongoDB!")

In [57]:

MAX_WORKERS = 5
URL_PATES_CONDIMENTS_CONSERVES = "https://www.migros.ch/fr/category/pates-condiments-conserves"
URL_PRODUITS_LAITIERS_OEUFS_PLATS = "https://www.migros.ch/fr/category/produits-laitiers-ufs-plats-prep"
prod = fetch_product_data(base_url=URL_PRODUITS_LAITIERS_OEUFS_PLATS,path_to_json="produits_laitiers_ufs_plats.json", max_workers=MAX_WORKERS, fetch_online=True)


Fetching online data from: https://www.migros.ch/fr/category/produits-laitiers-ufs-plats-prep
Setting up the browser...
Loading https://www.migros.ch/fr/category/produits-laitiers-ufs-plats-prep...
Waiting for products to load...
Total pages to scrape: 28
Thread starting for: https://www.migros.ch/fr/category/produits-laitiers-ufs-plats-prep?page=1
Setting up the browser...
Thread starting for: https://www.migros.ch/fr/category/produits-laitiers-ufs-plats-prep?page=2
Setting up the browser...
Thread starting for: https://www.migros.ch/fr/category/produits-laitiers-ufs-plats-prep?page=3
Setting up the browser...
Thread starting for: https://www.migros.ch/fr/category/produits-laitiers-ufs-plats-prep?page=4
Setting up the browser...
Thread starting for: https://www.migros.ch/fr/category/produits-laitiers-ufs-plats-prep?page=5
Setting up the browser...
Loading https://www.migros.ch/fr/category/produits-laitiers-ufs-plats-prep?page=2...
Loading https://www.migros.ch/fr/category/produits-lai

In [58]:
save_to_mongodb(prod, db_name="migros_db", collection_name="produits_laitiers_oeufs_plats")

Connected to MongoDB. Saving 2700 product snapshots...


C:\Users\B\AppData\Local\Temp\ipykernel_27368\670346365.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  scrape_timestamp = datetime.utcnow()


Snapshot data successfully saved to MongoDB!


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import requests
from io import BytesIO
from datetime import datetime

def generate_price_alert_image(product_data, output_filename):
    """
    Creates a Migros-style alert image from scratch using code.
    No 'template.png' required.
    """
    # 1. Canvas Settings
    width, height = 600, 800
    orange = (255, 102, 0)   # Migros Orange
    blue = (173, 216, 230)   # Light Blue
    white = (255, 255, 255)
    black = (0, 0, 0)
    red = (220, 20, 60)      # For 'Statt' price

    # Create a new image with white background
    img = Image.new("RGBA", (width, height), white)
    draw = ImageDraw.Draw(img)

    # 2. Draw Background Sections
    # Top Orange Header
    draw.rectangle([0, 0, width, 400], fill=orange)
    # Bottom Blue Product Area
    draw.rectangle([50, 450, 550, 750], fill=blue)

    # 3. Load Fonts
    try:
        # If on Windows: 'arial.ttf', If on Mac: '/Library/Fonts/Arial.ttf'
        font_main = ImageFont.truetype("arial.ttf", 110)
        font_sub = ImageFont.truetype("arial.ttf", 40)
        font_price = ImageFont.truetype("arial.ttf", 90)
        font_text = ImageFont.truetype("arial.ttf", 25)
    except:
        font_main = font_sub = font_price = font_text = ImageFont.load_default()

    # 4. Draw Branding
    date_str = datetime.now().strftime("%d.%m.%Y")
    draw.text((40, 40), f"{date_str} – ALERT", font=font_text, fill=white)
    draw.text((40, 100), "MIGRO", font=font_main, fill=white)
    
    # Draw the "Woche" box
    draw.rectangle([200, 230, 400, 300], fill=white)
    draw.text((220, 235), "Woche", font=font_sub, fill=orange)

    # 5. Paste Product Image from URL
    if product_data.get('image_url'):
        try:
            resp = requests.get(product_data['image_url'], timeout=5)
            p_img = Image.open(BytesIO(resp.content)).convert("RGBA")
            # Resize and center it in the blue box
            p_img.thumbnail((350, 350))
            img.paste(p_img, (125, 450), p_img)
        except Exception as e:
            print(f"Could not load image: {e}")

    # 6. Add Pricing and Info
    new_price = f"{product_data.get('price', 0):.2f}"
    old_price = f"{product_data.get('old_price', 0):.2f}"
    name = product_data.get('name', 'Product')

    # Draw Price Circle/Background
    draw.rectangle([60, 600, 300, 780], fill=white, outline=black, width=2)
    draw.text((80, 620), new_price, font=font_price, fill=black)
    draw.text((80, 720), f"statt {old_price}", font=font_text, fill=red)

    # Product Details
    draw.text((320, 620), name[:25], font=font_text, fill=black)
    if product_data.get('quantity'):
        draw.text((320, 655), product_data['quantity'], font=font_text, fill=black)
    
    # 7. Save
    img.save(output_filename)
    print(f"✅ Alert image saved to {output_filename}")


def get_product_history(product_id, db_name="migros_db", collection_name="produits_laitiers_oeufs_plats"):
    client = MongoClient("mongodb://localhost:27017/")
    collection = client[db_name][collection_name]

    # Fetch history for this ID, sorted oldest to newest
    history = list(collection.find({"id": product_id}).sort("scraped_at", 1))

    if not history:
        return None

    changes = []
    
    # Iterate starting from the second snapshot to compare with the one before it
    for i in range(1, len(history)):
        new = history[i]
        old = history[i-1]
        
        event_changes = {}

        # 1. Name Change (Brand rebranding or product description update)
        if old.get('name') != new.get('name'):
            event_changes["name"] = {"from": old.get('name'), "to": new.get('name')}

        # 2. Price Change
        if old.get('price') != new.get('price'):
            event_changes["price"] = {"from": old.get('price'), "to": new.get('price')}

        # 3. Quantity/Weight Change (The "Shrinkflation" Indicator)
        if old.get('quantity') != new.get('quantity'):
            event_changes["quantity"] = {"from": old.get('quantity'), "to": new.get('quantity')}

        # 4. Price Per Unit Change
        if old.get('price_per_unit') != new.get('price_per_unit'):
            event_changes["ppu"] = {"from": old.get('price_per_unit'), "to": new.get('price_per_unit')}

        # 5. Image Change
        if old.get('image_url') != new.get('image_url'):
            event_changes["image"] = "Update detected"

        # If any of the above triggered, record the event
        if event_changes:
            changes.append({
                "date": new['scraped_at'],
                "details": event_changes
            })
            filename = f"alert_{product_id}_{new['scraped_at'].strftime('%Y%m%d')}.png"
            generate_price_alert_image(new, filename)   

    return {
        "id": product_id,
        "current_name": history[-1].get('name'),
        "change_events": changes
    }

In [60]:
def get_all_product_ids(db_name="migros_db", collection_name="produits_laitiers_oeufs_plats"):
    """Returns a list of all unique product IDs in the collection."""
    client = MongoClient("mongodb://localhost:27017/")
    collection = client[db_name][collection_name]
    
    # .distinct('id') finds every unique value in the 'id' field
    unique_ids = collection.distinct("id")
    
    print(f"Found {len(unique_ids)} unique products in the database.")
    return unique_ids

def run_comprehensive_report():
    all_ids = get_all_product_ids() # From previous step
    
    print(f"Checking {len(all_ids)} products for internal changes...\n")
    
    for pid in all_ids:
        report = get_product_history(pid)
        
        if report and report['change_events']:
            print(f"🔔 Change detected for {report['current_name']} (ID: {pid})")
            
            for event in report['change_events']:
                date_str = event['date'].strftime("%Y-%m-%d")
                print(f"   [{date_str}]")
                
                for key, val in event['details'].items():
                    if key == "image":
                        print(f"      - Image was updated")
                    else:
                        print(f"      - {key.upper()}: {val['from']} -> {val['to']}")
            print("-" * 30)




In [61]:
run_comprehensive_report()

Found 2704 unique products in the database.
Checking 2704 products for internal changes...



ValueError: unknown file extension: 